# Modelado - Predicción de productividad

**Objetivo:** Entrenar y comparar varios modelos de regresión para predecir `productividad` usando los datos preprocesados.

**Pasos:**
1. Cargar datos de entrenamiento y prueba.
2. Entrenar modelos base: Regresión Lineal, Random Forest, XGBoost.
3. Evaluar con validación cruzada (RMSE, MAE, R²).
4. Ajustar hiperparámetros (Random Forest y XGBoost).
5. Evaluar el mejor modelo en el conjunto de prueba.
6. Guardar el modelo final.

In [ ]:
# Importar librerías y cargar datos procesados
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV

# Cargar los datos
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')

#Se aplica el metodo ravel porque el modelo espera un array para la variable objetivo
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

# Verificar dimensiones
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (2000, 15)
X_test: (500, 15)
y_train: (2000,)
y_test: (500,)


In [13]:
# Función para evaluar un modelo usando validación cruzada
# Parámetros:
#   - modelo: el algoritmo de ML a evaluar
#   - X: variables predictoras (DataFrame o array)
#   - y: variable objetivo (array 1D)
#   - cv: número de particiones (por defecto 5)
# Devuelve: RMSE, MAE y R² promedio (más su desviación estándar)

def evaluar_con_cv(modelo, X, y, cv=5):
    # Calculamos RMSE (raíz del error cuadrático medio) para cada fold
    rmse_scores = -cross_val_score(modelo, X, y, cv=cv, scoring='neg_root_mean_squared_error')
    # Calculamos MAE (error absoluto medio) para cada fold
    mae_scores = -cross_val_score(modelo, X, y, cv=cv, scoring='neg_mean_absolute_error')
    # Calculamos R² (coeficiente de determinación) para cada fold
    r2_scores = cross_val_score(modelo, X, y, cv=cv, scoring='r2')
    
    # Mostramos resultados promediados con su variabilidad (± desviación estándar)
    print(f"RMSE medio: {rmse_scores.mean():.3f} (± {rmse_scores.std():.3f})")
    print(f"MAE medio:  {mae_scores.mean():.3f} (± {mae_scores.std():.3f})")
    print(f"R² medio:   {r2_scores.mean():.3f} (± {r2_scores.std():.3f})")
    
    return rmse_scores.mean(), mae_scores.mean(), r2_scores.mean()

In [14]:
# Evaluar Regresión Lineal
print("REGRESIÓN LINEAL:")
modelo_lr = LinearRegression()
rmse_lr, mae_lr, r2_lr = evaluar_con_cv(modelo_lr, X_train, y_train)

REGRESIÓN LINEAL:
RMSE medio: 2.933 (± 0.080)
MAE medio:  2.347 (± 0.046)
R² medio:   0.604 (± 0.035)


In [15]:
# Evaluar Random Forest (con hiperparámetros por defecto)
print("RANDOM FOREST:")
modelo_rf = RandomForestRegressor(random_state=1000, n_jobs=-1)
rmse_rf, mae_rf, r2_rf = evaluar_con_cv(modelo_rf, X_train, y_train)

RANDOM FOREST:
RMSE medio: 2.944 (± 0.101)
MAE medio:  2.359 (± 0.073)
R² medio:   0.601 (± 0.039)


In [16]:
# Evaluar XGBoost (con hiperparámetros por defecto)
print("XGBOOST:")
modelo_xgb = XGBRegressor(random_state=1000, n_jobs=-1)
rmse_xgb, mae_xgb, r2_xgb = evaluar_con_cv(modelo_xgb, X_train, y_train)

XGBOOST:
RMSE medio: 2.985 (± 0.099)
MAE medio:  2.362 (± 0.084)
R² medio:   0.589 (± 0.047)


## Resultados de la comparación de modelos (validación cruzada)

Se evaluaron tres modelos de regresión utilizando validación cruzada con 5 particiones sobre el conjunto de entrenamiento (2000 observaciones). Las métricas obtenidas (promedio ± desviación estándar) son:

| Modelo | RMSE | MAE | R² |
|--------|------|-----|-----|
| Regresión Lineal | 2.933 ± 0.080 | 2.347 ± 0.046 | 0.604 ± 0.035 |
| Random Forest (por defecto) | 2.944 ± 0.101 | 2.359 ± 0.073 | 0.601 ± 0.039 |
| XGBoost (por defecto) | 2.985 ± 0.099 | 2.362 ± 0.084 | 0.589 ± 0.047 |

### Interpretación

- **RMSE (error cuadrático medio raíz):** aproximadamente 2.93 tareas/semana. Indica el error típico de las predicciones.
- **MAE (error absoluto medio):** aproximadamente 2.35 tareas/semana. Es el error medio absoluto, más fácil de interpretar.
- **R² (coeficiente de determinación):** ≈ 0.60. El modelo explica el 60% de la variabilidad de la productividad.

## Ajuste de hiperparámetros

Dado que los tres modelos mostraron rendimiento similar, se probará el ajuste de hiperparámetros de Random Forest y XGBoost para ver si se logra una mejora significativa sobre la regresión lineal. Si el RMSE no mejora sustancialmente, se mantendrá la regresión lineal como modelo final por su simplicidad.

In [ ]:
# Valores que probaremos para los hiperparámetros
param_grid_rf = {
    'n_estimators': [100, 200],      # número de árboles en el bosque
    'max_depth': [10, 20, None],     # profundidad máxima de cada árbol (None = sin límite)
    'min_samples_split': [2, 5]      # muestras mínimas para dividir un nodo
}

# Creamos el modelo base
rf = RandomForestRegressor(random_state=1, n_jobs=-1)

# Buscamos la mejor combinación con validación cruzada
grid_rf = GridSearchCV(rf, param_grid_rf, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_rf.fit(X_train, y_train)

# Recuperamos el mejor modelo encontrado
mejor_rf = grid_rf.best_estimator_

# Evaluamos ese mejor modelo con validación cruzada (5 folds por defecto) para obtener RMSE, MAE y R²
print("Random Forest ajustado:")
rmse_rf_adj, mae_rf_adj, r2_rf_adj = evaluar_con_cv(mejor_rf, X_train, y_train)

Random Forest ajustado:
RMSE medio: 2.920 (± 0.087)
MAE medio:  2.339 (± 0.065)
R² medio:   0.607 (± 0.036)


In [20]:
# Definimos los hiperparámetros a probar (valores comunes para empezar)
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Modelo base
xgb = XGBRegressor(random_state=1, n_jobs=-1)

# Búsqueda con validación cruzada 
grid_xgb = GridSearchCV(xgb, param_grid_xgb, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1)
grid_xgb.fit(X_train, y_train)

# Mejor modelo
mejor_xgb = grid_xgb.best_estimator_

# Evaluación con validación cruzada usando tu función
print("XGBoost ajustado:")
rmse_xgb_adj, mae_xgb_adj, r2_xgb_adj = evaluar_con_cv(mejor_xgb, X_train, y_train)

XGBoost ajustado:
RMSE medio: 2.693 (± 0.082)
MAE medio:  2.159 (± 0.066)
R² medio:   0.666 (± 0.034)


In [ ]:
# Evaluar el XGBoost ajustado en el conjunto de prueba
y_pred_xgb = mejor_xgb.predict(X_test)

rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_test = mean_absolute_error(y_test, y_pred_xgb)
r2_test = r2_score(y_test, y_pred_xgb)

print("XGBoost con datos de prueba")
print(f"RMSE: {rmse_test:.3f}")
print(f"MAE: {mae_test:.3f}")
print(f"R²: {r2_test:.3f}")

=== XGBoost ajustado - Evaluación en prueba ===
RMSE: 2.819
MAE: 2.285
R²: 0.634


## Evaluación final del modelo XGBoost ajustado

Se realizó un ajuste de hiperparámetros sobre XGBoost (búsqueda con validación cruzada sobre el conjunto de entrenamiento). El mejor modelo se evaluó sobre el conjunto de prueba (500 observaciones no vistas). Los resultados son:

| Métrica | Valor |
|--------|-------|
| **RMSE** | 2.819 (error típico en tareas/semana) |
| **MAE**  | 2.285 (error absoluto medio) |
| **R²**   | 0.634 (proporción de varianza explicada) |

**Interpretación:**  
El modelo explica el 63.4% de la variabilidad de la productividad. En promedio, las predicciones se desvían unas 2.3 tareas del valor real (MAE), con un error típico de 2.8 tareas (RMSE). Estos valores son ligeramente mejores que los obtenidos por la regresión lineal (R²=0.60, RMSE≈2.93). Por tanto, se selecciona XGBoost como modelo final para predecir la productividad de los empleados.

**Próximos pasos:**  
- Guardar el modelo entrenado (`joblib.dump(mejor_xgb, 'models/xgboost_final.pkl')`).
- Usar el modelo para predecir sobre nuevos datos (aplicando las mismas transformaciones logarítmicas, codificación y escalado).

In [22]:
# Guardar el modelo XGBoost ajustado en la carpeta models/
joblib.dump(mejor_xgb, '../models/xgboost_final.pkl')
print("Modelo guardado en models/xgboost_final.pkl")

Modelo guardado en models/xgboost_final.pkl


## Conclusión del proyecto

El proyecto de Machine Learning ha finalizado con éxito. Se completaron todas las fases planificadas:

- **EDA**: análisis exploratorio, limpieza y detección de sesgos.
- **ETL**: transformación logarítmica, codificación de categóricas, escalado y división train/test.
- **Modelado**: entrenamiento y comparación de Regresión Lineal, Random Forest y XGBoost.
- **Ajuste de hiperparámetros**: mejora del XGBoost, que resultó ser el mejor modelo.
- **Evaluación final en prueba**: RMSE=2.819, MAE=2.285, R²=0.634.
- **Guardado del modelo y el escalador** para su uso futuro.